In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.paths import INTERIM_DATA

## Importing call option and SPY index data

In [2]:
df_index = pd.read_parquet(INTERIM_DATA / '01-data-spy-vix-rf.parquet')
df_call  = pd.read_parquet(INTERIM_DATA / '02-data-spy-option.parquet')

display(df_call, df_index)

,date,expiration,T,k,iv,mid,spread,volume,open_interest,delta,gamma,theta,vega,rho
0,2013-01-02,2013-01-04,0.005479,120.0,0.18930,26.049999,0.22,10,140,1.00000,0.00000,-0.36340,0.00000,-1.00000
1,2013-01-02,2013-01-04,0.005479,121.0,0.18930,25.049999,0.22,10,0,1.00000,0.00000,-0.36640,0.00000,-1.00000
2,2013-01-02,2013-01-04,0.005479,122.0,0.18930,24.049999,0.22,0,0,1.00000,0.00000,-0.36940,0.00000,-1.00000
3,2013-01-02,2013-01-04,0.005479,123.0,0.18930,23.049999,0.22,0,10,1.00000,0.00000,-0.37240,0.00000,-1.00000
4,2013-01-02,2013-01-04,0.005479,124.0,0.18930,22.049999,0.22,0,0,1.00000,0.00000,-0.37550,0.00000,-1.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7069607,2026-03-04,2028-12-15,2.786301,1080.0,0.12745,3.500000,5.00,0,1,0.06032,0.00082,-0.01239,1.36752,1.05322
7069608,2026-03-04,2028-12-15,2.786301,1090.0,0.12613,3.000000,5.00,0,11,0.05323,0.00075,-0.01106,1.23923,0.93189
7069609,2026-03-04,2028-12-15,2.786301,1160.0,0.13814,2.505000,4.99,0,0,0.04228,0.00057,-0.00967,1.03041,0.73675
7069610,2026-03-04,2028-12-15,2.786301,1180.0,0.14027,2.285000,4.55,0,13,0.03852,0.00052,-0.00902,0.95545,0.67125


,date,spy_close_adj,spy_close,spy_high,spy_low,spy_open,spy_volume,rf,vixo,vixh,vixl,vix,spy_ret_adj,spy_ret
0,1993-01-29,24.175386,43.937500,43.968750,43.750000,43.968750,1003200,0.0001,12.490000,13.160000,12.420000,12.420000,NaN,NaN
1,1993-02-01,24.347321,44.250000,44.250000,43.968750,43.968750,480500,0.0001,12.510000,12.920000,12.180000,12.330000,0.007112,0.007112
2,1993-02-02,24.398920,44.343750,44.375000,44.125000,44.218750,201300,0.0001,12.470000,12.890000,12.220000,12.250000,0.002119,0.002119
3,1993-02-03,24.656834,44.812500,44.843750,44.375000,44.406250,529400,0.0001,11.980000,12.340000,11.790000,12.120000,0.010571,0.010571
4,1993-02-04,24.759987,45.000000,45.093750,44.468750,44.968750,531500,0.0001,11.860000,12.840000,11.690000,12.290000,0.004184,0.004184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8305,2026-01-26,690.843262,692.729980,694.130005,689.919983,690.489990,60473800,0.0002,16.900000,17.389999,15.800000,16.150000,0.005078,0.005078
8306,2026-01-27,693.595764,695.489990,696.530029,693.570007,694.179993,55506100,0.0002,16.020000,16.370001,15.740000,16.350000,0.003984,0.003984
8307,2026-01-28,693.525940,695.419983,697.840027,693.940002,697.049988,61172200,0.0002,16.090000,17.180000,16.049999,16.350000,-0.000101,-0.000101
8308,2026-01-29,692.149719,694.039978,697.059998,684.830017,696.390015,97486200,0.0002,16.040001,19.740000,16.020000,16.879999,-0.001984,-0.001984


## 11- Index features

In [3]:
df_index = df_index.sort_values('date').copy()

# prior-day VIX — causal feature, not same-day
df_index['vix_lag']     = df_index['vix'].shift(1)

# same-day VIX momentum (vix_t - vix_{t-1})
df_index['vix_mom']     = df_index['vix'] - df_index['vix_lag']

# lagged VIX momentum (vix_{t-1} - vix_{t-2})
df_index['vix_mom_lag'] = df_index['vix_mom'].shift(1)

## 12- Merging index & option chain

In [4]:
df = df_call.merge(df_index, on='date', how='left')

# check for unmatched dates
n_null_vix = df['vix'].isna().sum()

print(f'Rows with no index match (NaN vix): {n_null_vix:,}')
print(f'Merged shape: {df.shape}')
print(df.columns.to_list())

Rows with no index match (NaN vix): 98,289
Merged shape: (7069612, 30)
['date', 'expiration', 'T', 'k', 'iv', 'mid', 'spread', 'volume', 'open_interest', 'delta', 'gamma', 'theta', 'vega', 'rho', 'spy_close_adj', 'spy_close', 'spy_high', 'spy_low', 'spy_open', 'spy_volume', 'rf', 'vixo', 'vixh', 'vixl', 'vix', 'spy_ret_adj', 'spy_ret', 'vix_lag', 'vix_mom', 'vix_mom_lag']


## 13- Adding post-merge features

In [5]:
df = df.sort_values(['k', 'expiration', 'date'])

grp = ['k', 'expiration']

# log_moneyness: log(S / K) — positive = ITM, negative = OTM
df['log_moneyness'] = np.log(df['spy_close'] / df['k'])

# iv_lag: prior trading day IV for the same contract
df['iv_lag']   = df.groupby(grp)['iv'].shift(1)

# d_iv: daily IV change — target variable
df['d_iv']     = df['iv'] - df['iv_lag']

# d_iv_lag: prior day's IV change for the same contract
df['d_iv_lag'] = df.groupby(grp)['d_iv'].shift(1)

# log-scale liquidity features
df['log_oi']         = np.log1p(df['open_interest'])
df['log_volume']     = np.log1p(df['volume'])

# return-based features
df['abs_spy_ret']    = df['spy_ret'].abs()
df['ret_over_sqrtT'] = df['spy_ret'] / np.sqrt(df['T'])

print('NaN check after adding feature:')
print(df[['iv_lag', 'd_iv', 'd_iv_lag', 'vix_lag', 'vix_mom', 'vix_mom_lag']].isna().sum())

NaN check after adding feature:
iv_lag         225974
d_iv           225974
d_iv_lag       440220
vix_lag         98289
vix_mom         98289
vix_mom_lag     98289
dtype: int64


## 14- Drop rows without a valid d_iv or vix_lag

In [6]:
before = len(df)
df = df.dropna(subset=['d_iv', 'vix_lag'])
after = len(df)

print(f'Rows dropped (NaN d_iv or vix_lag) : {before - after:,}  ({(before-after)/before*100:.2f}%)')
print(f'Rows remaining                     : {after:,}')

Rows dropped (NaN d_iv or vix_lag) : 321,272  (4.54%)
Rows remaining                     : 6,748,340


## 15- Paper filters (Cao, Chen & Hull 2019)

In [7]:
df_filter = df.copy()

before = len(df_filter)

mask = (
    (df_filter['T']     >= 14/365)   &   # remove < 14 days to expiry
    (df_filter['delta'] >= 0.05)     &   # remove deep OTM
    (df_filter['delta'] <= 0.95)     &   # remove deep ITM
    (df_filter['iv']    <= 1.00)         # remove extreme IV
)

df_filter = df_filter[mask].copy()
after = len(df_filter)

print(f"Before: {before:,}")
print(f"After:  {after:,}")
print(f"Removed: {before - after:,} ({(before - after)/before*100:.1f}%)")
display(df_filter[['delta', 'T', 'iv', 'd_iv']].describe())
print(f"\nd_iv std ratio (before/after): {0.0764 / df_filter['d_iv'].std():.1f}x")

Before: 6,748,340
After:  3,788,641
Removed: 2,959,699 (43.9%)


,delta,T,iv,d_iv
count,3.788641e+06,3.788641e+06,3.788641e+06,3.788641e+06
mean,5.832310e-01,6.545441e-01,1.850694e-01,1.359260e-03
std,2.809299e-01,6.244642e-01,6.735344e-02,1.851659e-02
min,5.000000e-02,3.835616e-02,2.380000e-02,-3.893200e-01
25%,3.470800e-01,1.534247e-01,1.368300e-01,-2.279997e-03
50%,6.459400e-01,4.630137e-01,1.733400e-01,0.000000e+00
75%,8.362600e-01,9.095891e-01,2.216600e-01,2.590001e-03
max,9.500000e-01,3.000000e+00,9.995000e-01,8.376700e-01



d_iv std ratio (before/after): 4.1x


## 16- Feature selection and save

In [8]:
print(df.columns.to_list())
df.shape

['date', 'expiration', 'T', 'k', 'iv', 'mid', 'spread', 'volume', 'open_interest', 'delta', 'gamma', 'theta', 'vega', 'rho', 'spy_close_adj', 'spy_close', 'spy_high', 'spy_low', 'spy_open', 'spy_volume', 'rf', 'vixo', 'vixh', 'vixl', 'vix', 'spy_ret_adj', 'spy_ret', 'vix_lag', 'vix_mom', 'vix_mom_lag', 'log_moneyness', 'iv_lag', 'd_iv', 'd_iv_lag', 'log_oi', 'log_volume', 'abs_spy_ret', 'ret_over_sqrtT']


(6748340, 38)

In [9]:
col_ = [
    'date', 'expiration', 'T', 'k', 'iv', 'spy_ret',
    'delta', 'gamma', 'theta', 'vega', 'rho',
    'vix', 'vix_lag', 'vix_mom', 'vix_mom_lag', 'iv_lag', 'log_moneyness', 
    'd_iv_lag','log_oi', 'log_volume', 'abs_spy_ret', 'ret_over_sqrtT',
    # traget
    'd_iv'
]

df_final = df_filter[col_].copy().reset_index(drop=True)

### Note: before using d_iv_lag -> dropna(subset=['d_iv_lag'])

In [10]:
df_final['d_iv_lag'].isna().sum()

np.int64(89502)

In [11]:
# flaot 64 to float 32
float64_cols = df_final.select_dtypes(include=['float64']).columns
df_final[float64_cols] = df_final[float64_cols].astype('float32')

df_final = df_final.sort_values('date')

df_final.to_parquet(INTERIM_DATA / '03-data-merge-feature.parquet', index=False)